# Retriever Types Demo

Demonstrates all retriever types provided by `RetrieverFactory`:

| Type | Backend | Supports ingestion |
|------|---------|:-----------------:|
| `vector` | Chroma / InMemory / PGVector | ✓ |
| `bm25` | bm25s (disk-persisted) | ✓ |
| `ensemble` | weighted RRF fusion of any retrievers | ✓ (fans out) |
| `reranked` | any retriever + compression step | — |

All retriever configurations live in `config/baseline.yaml` under `retrievers:`.
The `RetrieverFactory.create("tag")` call does the rest.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys

from dotenv import load_dotenv
from rich import print  # noqa: F401
from rich.table import Table

assert load_dotenv(verbose=True)
sys.path.append(".")

## 1. Sample documents

A small corpus we'll ingest into each retriever.

In [ ]:
from langchain_core.documents import Document
from pathlib import Path

from genai_tk.core.retriever_factory import BM25DocumentStore, RetrieverFactory


DOCS = [
    Document(
        page_content="Vector search uses dense embeddings to find semantically similar text.",
        metadata={"source": "intro.md", "topic": "vector"},
    ),
    Document(
        page_content="BM25 is a classic keyword ranking function based on term frequency and inverse document frequency.",
        metadata={"source": "bm25.md", "topic": "bm25"},
    ),
    Document(
        page_content="Hybrid retrieval combines dense vector search with sparse keyword methods like BM25.",
        metadata={"source": "hybrid.md", "topic": "hybrid"},
    ),
    Document(
        page_content="Reranking re-scores an initial candidate set using a more expensive cross-encoder or embeddings similarity model.",
        metadata={"source": "reranking.md", "topic": "reranking"},
    ),
    Document(
        page_content="Reciprocal Rank Fusion (RRF) is a score fusion method used in ensemble retrievers to combine ranked lists.",
        metadata={"source": "rrf.md", "topic": "ensemble"},
    ),
    Document(
        page_content="LangChain's EnsembleRetriever normalises weights and applies RRF to merge results from multiple retrievers.",
        metadata={"source": "langchain.md", "topic": "ensemble"},
    ),
    Document(
        page_content="Chroma is an in-process vector database that stores embeddings on disk or in memory.",
        metadata={"source": "chroma.md", "topic": "vector"},
    ),
    Document(
        page_content="PostgreSQL with the pgvector extension supports both vector similarity and full-text search in a single SQL query.",
        metadata={"source": "pgvector.md", "topic": "postgres"},
    ),
]

print(f"Loaded {len(DOCS)} documents")

## 2. List available retriever configs

Everything declared under `retrievers:` in `config/baseline.yaml`.

In [ ]:


configs = RetrieverFactory.list_available_configs()
print(configs)

## 3. Vector retriever (`type: vector`)

Backed by an in-memory Chroma store. Supports metadata filters.

In [ ]:
vector = RetrieverFactory.create("default")
print("has_store:", vector.has_store)
print("stats:", vector.get_stats())

In [ ]:
# Ingest
await vector.aadd_documents(DOCS)

# Query
results = await vector.aquery("dense embeddings similarity", k=3)

table = Table(title="Vector results", show_lines=True)
table.add_column("#", width=3)
table.add_column("Content", max_width=80)
table.add_column("Source", style="dim")
for i, doc in enumerate(results, 1):
    table.add_row(str(i), doc.page_content, doc.metadata.get("source", ""))
print(table)

In [ ]:
# Metadata filter
filtered = await vector.aquery("search", k=5, filter={"topic": "vector"})
print(f"Filtered to topic=vector: {len(filtered)} result(s)")
for doc in filtered:
    print(f"  • {doc.metadata['source']}: {doc.page_content[:60]}…")

## 4. BM25 retriever (`type: bm25`)

Index built on-demand and persisted under `data/bm25_cache/bm25_local/`.

In [ ]:
bm25 = RetrieverFactory.create("bm25_local")
print("has_store:", bm25.has_store)
print("stats:", bm25.get_stats())

In [ ]:
# Build the index
await bm25.aadd_documents(DOCS)

# Query — keyword matching favours exact terms
results_bm25 = await bm25.aquery("BM25 keyword ranking function", k=3)

table = Table(title="BM25 results", show_lines=True)
table.add_column("#", width=3)
table.add_column("Content", max_width=80)
table.add_column("Source", style="dim")
for i, doc in enumerate(results_bm25, 1):
    table.add_row(str(i), doc.page_content, doc.metadata.get("source", ""))
print(table)

In [ ]:

print("get_stats()")
cache_dir = Path(bm25.get_stats()["bm25_cache_dir"])
print("Cache directory:", cache_dir)
print("Files:", [f.name for f in cache_dir.rglob("*") if f.is_file()])

In [ ]:
# BM25 cache survives restarts — demonstrate load from disk


fresh = BM25DocumentStore(cache_dir=cache_dir)
loaded = fresh.get_or_load_retriever(k=2)
print(f"Loaded index from disk — {len(loaded.docs) if loaded else 0} documents")

## 5. Ensemble retriever (`type: ensemble`)

Combines the vector and BM25 retrievers above using Reciprocal Rank Fusion (weights 0.7 / 0.3).

In [ ]:
ensemble = RetrieverFactory.create("hybrid_ensemble")
print("stats:", ensemble.get_stats())

In [ ]:
# Ensemble uses the underlying vector + BM25 retrievers; we've already indexed above.
results_ens = await ensemble.aquery("hybrid retrieval combining dense and sparse", k=4)

table = Table(title="Ensemble results (vector 0.7 + BM25 0.3)", show_lines=True)
table.add_column("#", width=3)
table.add_column("Content", max_width=80)
table.add_column("Source", style="dim")
for i, doc in enumerate(results_ens, 1):
    table.add_row(str(i), doc.page_content, doc.metadata.get("source", ""))
print(table)

## 6. Reranked retriever (`type: reranked`)

Wraps the ensemble with an `EmbeddingsFilter` (threshold 0.7). Fewer but higher-precision results.

In [ ]:
reranked = RetrieverFactory.create("hybrid_reranked")
print("stats:", reranked.get_stats())

results_rer = await reranked.aquery("combining ranked lists fusion", k=3)

table = Table(title="Reranked results", show_lines=True)
table.add_column("#", width=3)
table.add_column("Content", max_width=80)
table.add_column("Source", style="dim")
for i, doc in enumerate(results_rer, 1):
    table.add_row(str(i), doc.page_content, doc.metadata.get("source", ""))
print(table)

## 7. Side-by-side comparison

Same query against all four retriever types.

In [ ]:
QUERY = "how does hybrid search combine vector and keyword retrieval"

retrievers = {
    "vector": vector,
    "bm25": bm25,
    "ensemble": ensemble,
    "reranked": reranked,
}

for name, managed in retrievers.items():
    docs = await managed.aquery(QUERY, k=3)
    print(f"\n[bold cyan]{name}[/bold cyan] ({len(docs)} result(s))")
    for i, doc in enumerate(docs, 1):
        print(f"  {i}. [{doc.metadata.get('source', '?')}] {doc.page_content[:90]}…")

## 8. Programmatic config (no YAML)

You can also build a `ManagedRetriever` entirely in Python without touching the config file — useful for testing or dynamic construction.

In [ ]:
from pathlib import Path

from langchain_core.vectorstores import InMemoryVectorStore

from genai_tk.core.embeddings_factory import get_embeddings
from genai_tk.core.retriever_factory import ManagedRetriever, VectorDocumentStore

# Build an in-memory vector store directly
emb = get_embeddings()
vs = InMemoryVectorStore(embedding=emb)

store = VectorDocumentStore(vector_store=vs)
managed = ManagedRetriever(
    retriever=vs.as_retriever(search_kwargs={"k": 3}),
    store=store,
    default_k=3,
    config_tag="my_programmatic_store",
    vector_store=vs,
)

await managed.aadd_documents(DOCS[:4])
results = await managed.aquery("term frequency inverse document frequency")
print(f"Programmatic store: {len(results)} result(s)")
for doc in results:
    print(f"  • {doc.page_content[:80]}…")

## 9. Cleanup

In [ ]:
# Delete in-memory vector stores (no-op for BM25 to keep the cache for the RAG pipeline demo)
await vector.adelete_store()
print("Vector store cleared.")